# Causal Machine Learning — Entrega Intermediária
## Prevenção de Churn em Telecomunicações via Inferência Causal

**Grupo:** Eduardo Assis Tomich · Lucas Augusto da Silva Gonçalves · Lucas Vitor da Silva Ramos · Rafael Vinícius dos Santos

**Disciplina:** Causal Machine Learning — UFMG 2026/1

---

### Contexto do Problema

No setor de telecomunicações, o custo de aquisição de clientes (CAC) supera significativamente o de retenção, tornando o *churn* um desafio crítico. Modelos preditivos tradicionais identificam clientes com alto risco de cancelamento, mas **não prescrevem a ação ideal**.

O projeto completo formula a prevenção de churn como um problema de **Unit Selection** (Pearl, Nível 3), onde o objetivo é identificar *Compliers* — clientes que só permanecerão se receberem um tratamento específico.

### Formulação das Variáveis

| Símbolo | Descrição | Variáveis no Dataset |
|---------|-----------|---------------------|
| **C** (Covariáveis) | Atributos observáveis do cliente | `tenure`, `Contract`, `MonthlyCharges`, `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `InternetService`, etc. |
| **X** (Tratamentos) | Intervenções de retenção | `TechSupport`, `StreamingTV`, `StreamingMovies` |
| **Y** (Desfecho) | Retenção do cliente | `Churn` invertido: **Y=1 → Ficou**, **Y=0 → Cancelou** |

### Escopo desta Entrega

1. Limpeza e preparação dos dados
2. Análise Exploratória de Dados (EDA)
3. Modelos preditivos baseline (Regressão Logística + Random Forest)
4. DAG Causal do problema

---
## 1. Setup e Carregamento dos Dados

In [ ]:
%pip install -q kagglehub pandas numpy matplotlib seaborn scikit-learn networkx

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    average_precision_score,
    PrecisionRecallDisplay,
    brier_score_loss,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42

In [ ]:
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print(f"Dataset baixado em: {path}")

In [ ]:
import os

csv_file = os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_raw = pd.read_csv(csv_file)

print(f"Shape: {df_raw.shape}")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
df_raw.info()

---
## 2. Limpeza de Dados

Problemas conhecidos do dataset:
- `TotalCharges` está como `object` (strings vazias em clientes com `tenure=0`)
- `customerID` não é feature útil
- `Churn` precisa ser convertido para numérico com Y=1 representando **retenção**

In [ ]:
df = df_raw.copy()

# Remover customerID
df = df.drop(columns=["customerID"])

# Converter TotalCharges para numérico
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Verificar nulos gerados pela conversão
nulos_tc = df["TotalCharges"].isna().sum()
print(f"Valores nulos em TotalCharges após conversão: {nulos_tc}")
print(f"Esses clientes têm tenure = 0:")
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
# Clientes com tenure=0 e TotalCharges nulo: preencher com 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Criar variável target Y: 1 = Retenção (No Churn), 0 = Cancelou (Churn)
df["Y"] = (df["Churn"] == "No").astype(int)
df = df.drop(columns=["Churn"])

print(f"Distribuição de Y (1=Ficou, 0=Cancelou):")
print(df["Y"].value_counts())
print(f"\nNulos restantes por coluna:")
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("Nenhum valor nulo restante.")

In [ ]:
# Definir grupos de variáveis conforme a formulação do projeto
TREATMENTS = ["TechSupport", "StreamingTV", "StreamingMovies"]
TARGET = "Y"

COVARIATES = [
    col for col in df.columns if col not in TREATMENTS + [TARGET]
]

print(f"Covariáveis (C): {COVARIATES}")
print(f"Tratamentos (X): {TREATMENTS}")
print(f"Desfecho (Y): {TARGET}")

In [ ]:
df.describe()

In [ ]:
df.dtypes

---
## 3. Análise Exploratória de Dados (EDA)

### 3.1 Distribuição do Target (Retenção vs. Churn)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["Y"].value_counts()
labels = ["Cancelou (Y=0)", "Ficou (Y=1)"]
colors = ["#e74c3c", "#2ecc71"]
bars = ax.bar(labels, [counts[0], counts[1]], color=colors, edgecolor="black", width=0.5)

for bar, count in zip(bars, [counts[0], counts[1]]):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{count}\n({pct:.1f}%)", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Número de Clientes")
ax.set_title("Distribuição do Target — Desbalanceamento ~73/27%")
plt.tight_layout()
plt.show()

### 3.2 Features Demográficas vs. Retenção

In [ ]:
demo_features = ["gender", "SeniorCitizen", "Partner", "Dependents"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(demo_features):
    ax = axes[i]
    ct = pd.crosstab(df[feat], df["Y"], normalize="index") * 100
    ct.columns = ["Cancelou", "Ficou"]
    ct.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"], edgecolor="black", rot=0)
    ax.set_title(f"{feat} vs. Retenção")
    ax.set_ylabel("% dos Clientes")
    ax.set_xlabel(feat)
    ax.legend(title="Status", loc="upper right")
    ax.set_ylim(0, 100)

plt.suptitle("Features Demográficas vs. Retenção", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Features de Conta vs. Retenção

In [ ]:
num_features = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, feat in enumerate(num_features):
    ax = axes[i]
    for y_val, label, color in [(0, "Cancelou", "#e74c3c"), (1, "Ficou", "#2ecc71")]:
        subset = df[df["Y"] == y_val][feat]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor="black")
    ax.set_title(f"Distribuição de {feat}")
    ax.set_xlabel(feat)
    ax.set_ylabel("Frequência")
    ax.legend()

plt.suptitle("Features Numéricas por Status de Retenção", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, feat in enumerate(num_features):
    ax = axes[i]
    sns.kdeplot(data=df, x=feat, hue=df["Y"].map({0: "Cancelou", 1: "Ficou"}),
                fill=True, ax=ax, palette=["#e74c3c", "#2ecc71"])
    ax.set_title(f"KDE de {feat}")

plt.suptitle("Densidades das Features Numéricas", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Features de Serviço vs. Retenção

In [ ]:
service_features = ["InternetService", "Contract", "PaymentMethod", "PaperlessBilling"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(service_features):
    ax = axes[i]
    ct = pd.crosstab(df[feat], df["Y"], normalize="index") * 100
    ct.columns = ["Cancelou", "Ficou"]
    ct.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"], edgecolor="black", rot=30)
    ax.set_title(f"{feat} vs. Retenção")
    ax.set_ylabel("% dos Clientes")
    ax.set_xlabel(feat)
    ax.legend(title="Status")
    ax.set_ylim(0, 100)

plt.suptitle("Features de Serviço e Contrato vs. Retenção", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 3.5 Variáveis de Tratamento (X) vs. Retenção

Estas são as variáveis de tratamento do projeto: `TechSupport`, `StreamingTV`, `StreamingMovies`. Na entrega final, estimaremos o efeito causal (CATE) de cada uma sobre a retenção. Aqui analisamos a **associação observacional** — que inclui confundimento e não deve ser interpretada como efeito causal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, treat in enumerate(TREATMENTS):
    ax = axes[i]
    ct = pd.crosstab(df[treat], df["Y"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.columns = ["Cancelou", "Ficou"]

    ct_pct.plot(kind="bar", stacked=True, ax=ax,
                color=["#e74c3c", "#2ecc71"], edgecolor="black", rot=0)
    ax.set_title(f"{treat} vs. Retenção", fontweight="bold")
    ax.set_ylabel("% dos Clientes")
    ax.set_xlabel(treat)
    ax.legend(title="Status", loc="lower right")
    ax.set_ylim(0, 100)

    # Anotar contagens
    for j, (idx, row) in enumerate(ct_pct.iterrows()):
        total = ct.loc[idx].sum()
        ax.text(j, 50, f"n={total}", ha="center", va="center",
                fontsize=9, fontweight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6))

plt.suptitle("Variáveis de Tratamento (X) vs. Retenção — Associação Observacional",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de retenção bruta por tratamento (associação, não causal)
print("Taxa de retenção (Y=1) por valor de cada tratamento:\n")
for treat in TREATMENTS:
    rates = df.groupby(treat)["Y"].mean().round(3)
    print(f"--- {treat} ---")
    for val, rate in rates.items():
        print(f"  {val}: {rate:.1%} de retenção")
    print()

### 3.6 Matriz de Correlação

In [ ]:
# Codificar variáveis categóricas para a correlação
df_encoded = df.copy()
for col in df_encoded.select_dtypes(include="object").columns:
    df_encoded[col] = df_encoded[col].astype("category").cat.codes

fig, ax = plt.subplots(figsize=(16, 12))
corr = df_encoded.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Matriz de Correlação (variáveis codificadas)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Correlações mais fortes com Y
corr_with_y = corr["Y"].drop("Y").sort_values(key=abs, ascending=False)
print("Top correlações com Y (Retenção):\n")
print(corr_with_y.head(10).to_string())

---
## 4. Pré-processamento para Modelos de ML

Para os modelos preditivos baseline, usamos **todas as features** (C + X) para prever Y. O objetivo aqui é avaliar a capacidade preditiva, não o efeito causal — essa distinção será central na entrega final.

In [ ]:
# Separar features e target
X = df.drop(columns=[TARGET])
y = df[TARGET]

# One-hot encoding das categóricas
X = pd.get_dummies(X, drop_first=True)

print(f"Features após encoding: {X.shape[1]} colunas")
print(f"Exemplos: {list(X.columns[:10])} ...")

In [ ]:
# Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras (Y=1: {y_train.mean():.1%})")
print(f"Teste:  {X_test.shape[0]} amostras (Y=1: {y_test.mean():.1%})")

In [ ]:
# Padronizar features numéricas
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Features numéricas padronizadas.")
X_train[num_cols].describe().round(2)

---
## 5. Modelos Preditivos Baseline

Treinamos dois modelos simples para estabelecer um baseline preditivo. As métricas escolhidas seguem a seção 9 do projeto:

- **PR-AUC** (Precision-Recall AUC): preferível ao ROC-AUC devido ao desbalanceamento. Calculado em relação à classe minoritária (Cancelou, Y=0).
- **Brier Score**: avalia a calibração das probabilidades — fundamental para a estimação de CATE na entrega final.

### 5.1 Regressão Logística

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

# PR-AUC calculado para a classe minoritária (Y=0 = Cancelou)
# average_precision_score por padrão calcula para pos_label=1,
# então invertemos para focar na classe minoritária
pr_auc_lr = average_precision_score(y_test, 1 - y_prob_lr, pos_label=0)
brier_lr = brier_score_loss(y_test, y_prob_lr)

print("=" * 50)
print("REGRESSÃO LOGÍSTICA")
print("=" * 50)
print(f"PR-AUC (classe minoritária): {pr_auc_lr:.4f}")
print(f"Brier Score:                 {brier_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=["Cancelou (0)", "Ficou (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=["Cancelou (0)", "Ficou (1)"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Regressão Logística — Matriz de Confusão")

# Curva Precision-Recall
PrecisionRecallDisplay.from_predictions(
    y_test, y_prob_lr, pos_label=1,
    name="Logistic Regression", ax=axes[1]
)
axes[1].set_title("Regressão Logística — Curva Precision-Recall")

plt.tight_layout()
plt.show()

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

pr_auc_rf = average_precision_score(y_test, 1 - y_prob_rf, pos_label=0)
brier_rf = brier_score_loss(y_test, y_prob_rf)

print("=" * 50)
print("RANDOM FOREST")
print("=" * 50)
print(f"PR-AUC (classe minoritária): {pr_auc_rf:.4f}")
print(f"Brier Score:                 {brier_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=["Cancelou (0)", "Ficou (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=["Cancelou (0)", "Ficou (1)"],
    cmap="Greens", ax=axes[0]
)
axes[0].set_title("Random Forest — Matriz de Confusão")

PrecisionRecallDisplay.from_predictions(
    y_test, y_prob_rf, pos_label=1,
    name="Random Forest", ax=axes[1]
)
axes[1].set_title("Random Forest — Curva Precision-Recall")

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
top15.plot(kind="barh", ax=ax, color="#3498db", edgecolor="black")
ax.set_title("Top 15 Feature Importances — Random Forest", fontweight="bold")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.show()

### 5.3 Comparação dos Modelos

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Regressão Logística", "Random Forest"],
    "PR-AUC (minoritária)": [pr_auc_lr, pr_auc_rf],
    "Brier Score": [brier_lr, brier_rf],
}).set_index("Modelo")

print("Comparação dos Modelos Baseline:")
print("(PR-AUC: maior é melhor | Brier Score: menor é melhor)")
print()
comparison.round(4)

---
## 6. DAG Causal

O grafo acíclico direcionado (DAG) abaixo representa a estrutura causal assumida para o problema de churn em telecomunicações, conforme a formulação do projeto.

### Estrutura do DAG

- **C → X**: As covariáveis do cliente (perfil demográfico, tipo de contrato, tempo de casa) influenciam quais serviços o cliente contrata — gerando **viés de seleção** nos dados observacionais.
- **C → Y**: As mesmas covariáveis afetam diretamente a probabilidade de retenção (ex: clientes com contratos longos têm menor churn).
- **X → Y**: Os tratamentos (TechSupport, StreamingTV, StreamingMovies) têm um **efeito causal direto** sobre a retenção — este é o efeito que queremos estimar.

### Premissa de Unconfoundedness

A premissa central é que, **condicionando nas covariáveis C observadas**, a atribuição do tratamento X é independente dos desfechos potenciais. Ou seja, não há confundidores não-observados. Essa premissa habilita o **Backdoor Adjustment**: ao controlar por C, podemos estimar $P(Y | do(X))$ a partir dos dados observacionais.

Na entrega final, utilizaremos Propensity Score Matching (PSM) ou IPW para operacionalizar esse ajuste, e os testes de refutação do DoWhy para avaliar a sensibilidade à violação dessa premissa.

In [ ]:
G = nx.DiGraph()

# -- Nós --
covariates_nodes = [
    "tenure", "Contract", "MonthlyCharges",
    "SeniorCitizen", "gender", "Partner",
    "Dependents", "InternetService",
    "PaymentMethod", "PaperlessBilling",
    "PhoneService", "MultipleLines",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
]
treatment_nodes = ["TechSupport", "StreamingTV", "StreamingMovies"]
outcome_node = "Retenção (Y)"

G.add_nodes_from(covariates_nodes)
G.add_nodes_from(treatment_nodes)
G.add_node(outcome_node)

# -- Arestas --
# C → X (covariáveis influenciam a adoção dos tratamentos)
confounders_to_treatment = [
    "tenure", "Contract", "MonthlyCharges", "SeniorCitizen",
    "InternetService", "Partner", "Dependents"
]
for c in confounders_to_treatment:
    for x in treatment_nodes:
        G.add_edge(c, x)

# C → Y (covariáveis influenciam a retenção diretamente)
for c in covariates_nodes:
    G.add_edge(c, outcome_node)

# X → Y (efeito causal dos tratamentos sobre a retenção)
for x in treatment_nodes:
    G.add_edge(x, outcome_node)

# Algumas relações entre covariáveis
G.add_edge("tenure", "TotalCharges" if "TotalCharges" in G.nodes else "MonthlyCharges")
G.add_edge("InternetService", "OnlineSecurity")
G.add_edge("InternetService", "OnlineBackup")
G.add_edge("InternetService", "DeviceProtection")
G.add_edge("PhoneService", "MultipleLines")

# -- Layout --
pos = {}

# Covariáveis em duas fileiras no topo
top_row = ["gender", "SeniorCitizen", "Partner", "Dependents",
           "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup"]
mid_row = ["tenure", "Contract", "MonthlyCharges", "InternetService",
           "PaymentMethod", "PaperlessBilling", "DeviceProtection"]

for i, node in enumerate(top_row):
    pos[node] = (i * 1.8, 3)
for i, node in enumerate(mid_row):
    pos[node] = (i * 2.1 + 0.5, 1.5)

# Tratamentos no meio
for i, node in enumerate(treatment_nodes):
    pos[node] = (3 + i * 3.5, 0)

# Desfecho embaixo
pos[outcome_node] = (6.5, -1.5)

# -- Cores por tipo de nó --
node_colors = []
for node in G.nodes():
    if node in treatment_nodes:
        node_colors.append("#f39c12")
    elif node == outcome_node:
        node_colors.append("#2ecc71")
    else:
        node_colors.append("#3498db")

# -- Desenhar --
fig, ax = plt.subplots(figsize=(20, 10))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2000,
                       edgecolors="black", linewidths=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold", ax=ax)

# Arestas C→Y em cinza claro, X→Y em vermelho forte, C→X em laranja
edge_colors = []
edge_widths = []
for u, v in G.edges():
    if u in treatment_nodes and v == outcome_node:
        edge_colors.append("#e74c3c")
        edge_widths.append(2.5)
    elif v in treatment_nodes:
        edge_colors.append("#f39c12")
        edge_widths.append(1.2)
    else:
        edge_colors.append("#bdc3c7")
        edge_widths.append(0.8)

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       arrows=True, arrowsize=15, arrowstyle="->",
                       connectionstyle="arc3,rad=0.05", ax=ax)

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#3498db", edgecolor="black", label="Covariáveis (C)"),
    Patch(facecolor="#f39c12", edgecolor="black", label="Tratamentos (X)"),
    Patch(facecolor="#2ecc71", edgecolor="black", label="Desfecho (Y)"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=12,
          framealpha=0.9, edgecolor="black")

ax.set_title("DAG Causal — Prevenção de Churn em Telecomunicações\n"
             "Arestas vermelhas = efeito causal de interesse (X → Y) | "
             "Laranjas = viés de seleção (C → X)",
             fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

### Interpretação do DAG

As arestas **vermelhas** (X → Y) representam o efeito causal que queremos isolar na entrega final: o impacto de oferecer TechSupport, StreamingTV ou StreamingMovies sobre a retenção.

As arestas **laranjas** (C → X) representam o **viés de seleção**: clientes com certas características (ex: maior tenure, contrato longo) têm maior probabilidade de já possuir esses serviços. Se não controlarmos por C, a associação observacional entre X e Y mistura o efeito causal com esse viés.

O critério **Backdoor** nos diz que, ao condicionar em C, bloqueamos todos os caminhos espúrios entre X e Y, permitindo identificar o efeito causal $P(Y | do(X))$ a partir dos dados observacionais.

---
## 7. Conclusões Parciais e Próximos Passos

### Achados da EDA
- O dataset apresenta desbalanceamento significativo (~73% retenção vs. ~27% churn)
- `tenure`, `Contract` e `MonthlyCharges` são os preditores mais fortes de retenção
- Clientes com Fiber Optic e contratos mês-a-mês apresentam as maiores taxas de cancelamento
- A associação observacional sugere que TechSupport tem forte relação com retenção, mas esta inclui confundimento (clientes que contratam TechSupport tendem a ter contratos mais longos)

### Modelos Baseline
- Ambos os modelos (Regressão Logística e Random Forest) foram treinados e avaliados com PR-AUC e Brier Score, conforme definido na proposta
- Esses estimadores-base servirão como componentes dos Meta-Learners na próxima etapa

### Próximos Passos (Entrega Final)
1. **Propensity Score Matching / IPW**: ajuste observacional para controlar o viés de seleção
2. **Meta-Learners (T-Learner / X-Learner)**: estimação do CATE individual para cada tratamento
3. **Unit Selection**: classificação dos clientes nos estratos causais (Always-takers, Never-takers, Defiers, Compliers)
4. **Otimização de Política**: seleção do tratamento ótimo por perfil de cliente
5. **Testes de Refutação (DoWhy)**: placebo treatment e sensitivity analysis